In [1]:
import os

# Caminho absoluto da pasta onde o script está sendo executado
diretorio_atual = os.getcwd()

print("Diretório atual:", diretorio_atual)

Diretório atual: /workspaces/IC-RNA-2025/Peaks-dataset-article/media_ponderada


In [3]:
import pandas as pd
import numpy as np
import ast
from sklearn.metrics import mean_squared_error

# ================================
# 1) CONFIGURAÇÕES
# ================================
arquivo_pesos = "/workspaces/IC-RNA-2025/Peaks-dataset-article/media_ponderada/todos_resultados_50_lm4_novo.xlsx"
diretorio_redes = "/workspaces/IC-RNA-2025/Peaks-dataset-article/media_ponderada/1000_model"
limiar_peso = 0.1

df_pesos = pd.read_excel(arquivo_pesos).head(50)

resultados = []

for idx, row in df_pesos.iterrows():
    # Coluna A = nomes das redes, Coluna E = pesos
    nomes_str = row.iloc[0]
    pesos_str = row.iloc[4]

    # 🔧 Converte string "[a b c]" → lista de strings → lista de floats
    nomes_redes = nomes_str.strip("[]").replace("'", "").split(",")
    nomes_redes = [nome.strip() for nome in nomes_redes]

    pesos = np.array(pesos_str.strip("[]").split(), dtype=float)


    # ===============================
    # 2) CARREGA Z E Z_pred
    # ===============================
    dfs = [pd.read_excel(f"{diretorio_redes}/{nome}") for nome in nomes_redes]
    Z = dfs[0]["Z"].values.reshape(-1, 1)
    Z_preds = np.hstack([df["Z_pred"].values.reshape(-1, 1) for df in dfs])

    # ===============================
    # 3) MSE COM TODAS AS REDES
    # ===============================
    Z_pred_ponderada = Z_preds @ pesos
    mse_sup = mean_squared_error(Z, Z_pred_ponderada)

    # ===============================
    # 4) REMOVE REDES COM PESO < 0.1
    # ===============================
    mask = pesos >= 0.1
    nomes_filtrados = [nome for nome, keep in zip(nomes_redes, mask) if keep]
    pesos_filtrados = pesos[mask]
    pesos_filtrados = pesos_filtrados / np.sum(pesos_filtrados)

    Z_preds_filtrados = Z_preds[:, mask]
    yhat_filtrado = Z_preds_filtrados @ pesos_filtrados
    mse_filtrado = mean_squared_error(Z, yhat_filtrado)

    resultados.append({
        "linha": idx + 1,
        "MSE_sup": mse_sup,
        "MSE_sup_novo": mse_filtrado,
        "num_redes_total": len(nomes_redes),
        "num_redes_restantes": len(nomes_filtrados)
    })

# ===============================
# 5) RESULTADOS EM DATAFRAME
# ===============================
df_resultados = pd.DataFrame(resultados)
print(df_resultados)

# Salva se quiser
df_resultados.to_excel("comparacao_mse_50_ensembles.xlsx", index=False)

    linha   MSE_sup  MSE_sup_novo  num_redes_total  num_redes_restantes
0       1  0.573259      0.573259               10                    4
1       2  0.575100      0.571775               10                    4
2       3  0.577561      0.577561               10                    4
3       4  0.582700      0.589056               10                    3
4       5  0.583078      0.583078               10                    3
5       6  0.583583      0.583583               10                    4
6       7  0.586029      0.585652               10                    5
7       8  0.586309      0.577259               10                    5
8       9  0.587189      0.582654               10                    2
9      10  0.587423      0.586657               10                    3
10     11  0.587472      0.587472               10                    4
11     12  0.587619      0.587619               10                    6
12     13  0.588171      0.598099               10              